# Bankruptcy Prediction

## Data loader

In [ ]:
# Library Imports
import pandas as pd
import re

In [24]:
# Calling the data loader and preprocessing script to load the data and perform necessary preprocessing steps for bankruptcy prediction analysis.
%run data_loader_panel.py

Loaded: df_active (Active_companies.xlsx)

DataFrame: df_active

  df_active
  Rows: 9,100    Columns: 108

|   # | Column                            | Dtype   |   Non-Null |   Null |
|-----|-----------------------------------|---------|------------|--------|
|   0 | Company name Latin alphabet       | str     |      9,100 |      0 |
|   1 | NACE Rev. 2, core code (4 digits) | str     |      9,100 |      0 |
|   2 | P/L before tax EUR 2024           | Float64 |      8,719 |    381 |
|   3 | P/L before tax EUR 2023           | Float64 |      8,629 |    471 |
|   4 | P/L before tax EUR 2022           | Float64 |      8,315 |    785 |
|   5 | P/L before tax EUR 2021           | Float64 |      7,926 |  1,174 |
|   6 | P/L before tax EUR 2020           | Float64 |      7,497 |  1,603 |
|   7 | P/L before tax EUR 2019           | Float64 |      7,074 |  2,026 |
|   8 | P/L before tax EUR 2018           | Float64 |      6,637 |  2,463 |
|   9 | P/L before tax EUR 2017           | Float64 |   

## Data loading and preprocessing 

In [26]:
# ── 1. Label and merge ───────────────────────────────────────────────────────
df_active["Bankrupt"]   = 0
df_bankrupt["Bankrupt"] = 1

df_analysis = pd.concat([df_active, df_bankrupt], ignore_index=True)

print(f"After concat: {len(df_analysis):,} rows x {len(df_analysis.columns)} columns")
print(f"  Active:   {(df_analysis['Bankrupt'] == 0).sum():,}")
print(f"  Bankrupt: {(df_analysis['Bankrupt'] == 1).sum():,}")

# ── 2. NACE section mapping ──────────────────────────────────────────────────
nace_section_map = {
    range(1,  4):  "A - Agriculture, forestry, fishing",
    range(5,  10): "B - Mining and quarrying",
    range(10, 34): "C - Manufacturing",
    range(36, 40): "E - Water supply, waste management",
    range(41, 44): "F - Construction",
    range(45, 48): "G - Wholesale and retail trade",
    range(49, 54): "H - Transport and storage",
    range(55, 57): "I - Accommodation and food service",
    range(58, 64): "J - Information and communication",
    range(68, 69): "L - Real estate",
    range(69, 76): "M - Professional, scientific, technical",
    range(77, 83): "N - Administrative and support services",
    range(90, 94): "R - Arts, entertainment, recreation",
    range(94, 97): "S - Other service activities",
}

def nace_to_section(code):
    try:
        division = int(str(code)[:2])
        for r, section in nace_section_map.items():
            if division in r:
                return section
        return "Unknown"
    except:
        return "Unknown"

df_analysis["NACE Section"] = (
    df_analysis["NACE Rev. 2, core code (4 digits)"]
    .apply(nace_to_section)
)

# Quick check on unknown NACE codes
unknown_nace = df_analysis[df_analysis["NACE Section"] == "Unknown"]
if len(unknown_nace) > 0:
    print(f"\nUnknown NACE codes ({len(unknown_nace)} companies):")
    print(unknown_nace["NACE Rev. 2, core code (4 digits)"].value_counts().head(10))
else:
    print("✓ All NACE codes mapped")

# ── 3. Incorporation year ────────────────────────────────────────────────────
df_analysis["Year of incorporation"] = pd.to_numeric(
    df_analysis["Year of incorporation"], errors="coerce"
).astype("Int64")

# ── 4. Column ordering ───────────────────────────────────────────────────────
front_cols = [
    "Company name Latin alphabet",
    "Year of incorporation",
    "Bankrupt",
    "NACE Section",
    "NACE Rev. 2, core code (4 digits)",
]

remaining_cols = [c for c in df_analysis.columns if c not in front_cols]
df_analysis = df_analysis[front_cols + remaining_cols]

print_df_structure("df_analysis", df_analysis)

After concat: 10,236 rows x 109 columns
  Active:   9,100
  Bankrupt: 1,136
✓ All NACE codes mapped

DataFrame: df_analysis

  df_analysis
  Rows: 10,236    Columns: 110

|   # | Column                            | Dtype   |   Non-Null |   Null |
|-----|-----------------------------------|---------|------------|--------|
|   0 | Company name Latin alphabet       | str     |     10,236 |      0 |
|   1 | Year of incorporation             | Int64   |     10,235 |      1 |
|   2 | Bankrupt                          | int64   |     10,236 |      0 |
|   3 | NACE Section                      | str     |     10,236 |      0 |
|   4 | NACE Rev. 2, core code (4 digits) | str     |     10,236 |      0 |
|   5 | P/L before tax EUR 2024           | Float64 |      9,172 |  1,064 |
|   6 | P/L before tax EUR 2023           | Float64 |      9,415 |    821 |
|   7 | P/L before tax EUR 2022           | Float64 |      9,345 |    891 |
|   8 | P/L before tax EUR 2021           | Float64 |      8,895 |  1

In [27]:
from IPython.display import display, HTML
import pandas as pd

def display_scrollable(obj, height=400, na_rep=""):
    if isinstance(obj, pd.DataFrame):
        html = obj.to_html(classes="scrollable-table", na_rep=na_rep)
    else:
        html = "<pre>{}</pre>".format(HTML.escape(str(obj)))
    style = f"""
    <style>
      .scrollable-table-wrapper {{ max-height: {height}px; overflow: auto; border:1px solid #ddd; }}
      .scrollable-table {{ border-collapse: collapse; width:100%; }}
    </style>
    """
    display(HTML(style + f'<div class="scrollable-table-wrapper">{html}</div>'))

# Example: show the first 200 rows of df_analysis in a scrollable pane
display_scrollable(df_analysis.head(200), height=500)

,Company name Latin alphabet,Year of incorporation,Bankrupt,NACE Section,"NACE Rev. 2, core code (4 digits)",P/L before tax EUR 2024,P/L before tax EUR 2023,P/L before tax EUR 2022,P/L before tax EUR 2021,P/L before tax EUR 2020,P/L before tax EUR 2019,P/L before tax EUR 2018,P/L before tax EUR 2017,P/L before tax EUR 2016,P/L before tax EUR 2015,P/L before tax EUR 2014,P/L before tax EUR 2013,P/L before tax EUR 2012,P/L before tax EUR 2011,P/L before tax EUR 2010,P/L before tax EUR 2009,P/L before tax EUR 2008,P/L before tax EUR 2007,P/L before tax EUR 2006,P/L before tax EUR 2005,P/L before tax EUR 2004,Total assets EUR 2024,Total assets EUR 2023,Total assets EUR 2022,Total assets EUR 2021,Total assets EUR 2020,Total assets EUR 2019,Total assets EUR 2018,Total assets EUR 2017,Total assets EUR 2016,Total assets EUR 2015,Total assets EUR 2014,Total assets EUR 2013,Total assets EUR 2012,Total assets EUR 2011,Total assets EUR 2010,Total assets EUR 2009,Total assets EUR 2008,Total assets EUR 2007,Total assets EUR 2006,Total assets EUR 2005,Total assets EUR 2004,Current liabilities EUR 2024,Current liabilities EUR 2023,Current liabilities EUR 2022,Current liabilities EUR 2021,Current liabilities EUR 2020,Current liabilities EUR 2019,Current liabilities EUR 2018,Current liabilities EUR 2017,Current liabilities EUR 2016,Current liabilities EUR 2015,Current liabilities EUR 2014,Current liabilities EUR 2013,Current liabilities EUR 2012,Current liabilities EUR 2011,Current liabilities EUR 2010,Current liabilities EUR 2009,Current liabilities EUR 2008,Current liabilities EUR 2007,Current liabilities EUR 2006,Current liabilities EUR 2005,Current liabilities EUR 2004,Current assets EUR 2024,Current assets EUR 2023,Current assets EUR 2022,Current assets EUR 2021,Current assets EUR 2020,Current assets EUR 2019,Current assets EUR 2018,Current assets EUR 2017,Current assets EUR 2016,Current assets EUR 2015,Current assets EUR 2014,Current assets EUR 2013,Current assets EUR 2012,Current assets EUR 2011,Current assets EUR 2010,Current assets EUR 2009,Current assets EUR 2008,Current assets EUR 2007,Current assets EUR 2006,Current assets EUR 2005,Current assets EUR 2004,Shareholders funds EUR 2024,Shareholders funds EUR 2023,Shareholders funds EUR 2022,Shareholders funds EUR 2021,Shareholders funds EUR 2020,Shareholders funds EUR 2019,Shareholders funds EUR 2018,Shareholders funds EUR 2017,Shareholders funds EUR 2016,Shareholders funds EUR 2015,Shareholders funds EUR 2014,Shareholders funds EUR 2013,Shareholders funds EUR 2012,Shareholders funds EUR 2011,Shareholders funds EUR 2010,Shareholders funds EUR 2009,Shareholders funds EUR 2008,Shareholders funds EUR 2007,Shareholders funds EUR 2006,Shareholders funds EUR 2005,Shareholders funds EUR 2004
0,JANGLAS OY,1998,0,C - Manufacturing,2312,-14000.0,-12000.0,44000.0,17000.0,45000.0,48000.0,9000.0,32000.0,13000.0,1000.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,159000.0,220000.0,231000.0,167000.0,157000.0,118000.0,90000.0,88000.0,34000.0,34000.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,55000.0,85000.0,65000.0,50000.0,47000.0,38000.0,45000.0,47000.0,21000.0,29000.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,133000.0,187000.0,190000.0,145000.0,142000.0,102000.0,86000.0,84000.0,29000.0,29000.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,93000.0,116000.0,140000.0,114000.0,109000.0,79000.0,44000.0,40000.0,12000.0,-2000.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,PERNAJAN RAKENNUS OY,2008,0,F - Construction,4120,325000.0,260000.0,190000.0,288712.0,28000.0,235000.0,159000.0,348000.0,258000.0,227000.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1947000.0,1699000.0,1665000.0,1584617.0,1426000.0,1582000.0,1418000.0,1478000.0,1206000.0,1129000.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,169000.0,80000.0,154000.0,115495.0,71000.0,150000.0,172000.0,240000.0,155000.0,188000.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<

In [28]:
# Verify
# Verify
print(df_analysis.groupby("NACE Section")["Bankrupt"].value_counts().unstack(fill_value=0))
print(df_analysis["Bankrupt"].value_counts())
print(df_analysis["Bankrupt"].value_counts(normalize=True).round(3))
print(df_analysis["Bankrupt"].value_counts(normalize=True).round(3))

Bankrupt                                    0    1
NACE Section                                      
A - Agriculture, forestry, fishing        346   21
B - Mining and quarrying                   38    1
C - Manufacturing                        1029  117
E - Water supply, waste management         79    5
F - Construction                         1483  305
G - Wholesale and retail trade           1684  220
H - Transport and storage                 677   88
I - Accommodation and food service        516   91
J - Information and communication         499   31
L - Real estate                           750   32
M - Professional, scientific, technical  1160   79
N - Administrative and support services   547  106
R - Arts, entertainment, recreation       187   23
S - Other service activities              105   17
Bankrupt
0    9100
1    1136
Name: count, dtype: int64
Bankrupt
0    0.889
1    0.111
Name: proportion, dtype: float64
Bankrupt
0    0.889
1    0.111
Name: proportion, dtype: float64
